# Clone Branch + Train FinBERT on Colab

This notebook needs **no repo upload**. It clones the public GitHub repo, checks out the project branch, uses the dataset already in Google Drive, and trains the FinBERT multimodal model.

Expected setup:

- Public repo: `https://github.com/pranavgupta55/2450FinalProject`
- Branch: `improve-model-training-pipeline`
- Dataset folder in Drive: `MyDrive/CIS2450/sp500_500_2yr`
- Colab runtime: **T4 GPU**

The model is evaluated as **positive-only / long-only**, matching the current project semantics.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 1. Configuration

Only edit these variables if your branch name or Drive dataset path changes.

Important: `MAX_LENGTH` must stay at or below `512` for `ProsusAI/finbert`, because FinBERT is BERT-based and cannot accept 1024-token sequences without a chunking redesign.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/pranavgupta55/2450FinalProject.git'
BRANCH = 'improve-model-training-pipeline'
WORKDIR = Path('/content/2450FinalProject')

DATASET_NAME = 'sp500_500_2yr'
DATASET_SOURCE = Path('/content/drive/MyDrive/CIS2450/sp500_500_2yr')
DATASET_TARGET = WORKDIR / 'data' / DATASET_NAME

HF_CACHE_DIR = Path('/content/drive/MyDrive/CIS2450/hf_cache')
ARTIFACT_BACKUP_DIR = Path('/content/drive/MyDrive/CIS2450/finbert_colab_artifacts')

EPOCHS = 10
BATCH_SIZE = 8
MAX_LENGTH = 512
LEARNING_RATE = 2e-4
CHECKPOINT_METRIC = 'average_precision'
THRESHOLD_METRIC = 'f1'
ALPHA_TRADE_OBJECTIVE = 'return_on_traded_capital'
INCLUDE_TICKER = True
TRAIN_ON_TEXT_ONLY = False
LOCAL_FILES_ONLY = False
UNFREEZE_FINBERT = False

assert MAX_LENGTH <= 512, 'ProsusAI/finbert only supports max_length <= 512.'
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print('Repo:', REPO_URL)
print('Branch:', BRANCH)
print('Dataset source:', DATASET_SOURCE)
print('Workdir:', WORKDIR)

## 2. Clone the branch and copy the Drive dataset locally

Colab training is faster from `/content` than directly from Drive, so this copies your Drive dataset into the cloned repo's `data/` folder.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

clone_cmd = [
    'git', 'clone',
    '--branch', BRANCH,
    '--single-branch',
    REPO_URL,
    str(WORKDIR),
]
print('Running:', ' '.join(clone_cmd))
subprocess.run(clone_cmd, check=True)

if not DATASET_SOURCE.exists():
    raise FileNotFoundError(f'Dataset folder not found: {DATASET_SOURCE}')

if DATASET_TARGET.exists():
    shutil.rmtree(DATASET_TARGET)
DATASET_TARGET.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DATASET_SOURCE, DATASET_TARGET)

os.chdir(WORKDIR)
print('Current working directory:', Path.cwd())
print('Dataset copied to:', DATASET_TARGET)
print('Dataset files:', sorted(p.name for p in DATASET_TARGET.iterdir())[:20])

## 3. Install dependencies

If Colab asks you to restart after install, restart the runtime and rerun cells from the top.

In [ ]:
%pip install -q --upgrade pip
%pip install -q -r requirements.txt
%pip install -q sentencepiece accelerate

## 4. Check GPU and dataset coverage

In [ ]:
import json
import os
from pathlib import Path

import torch

os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE_DIR)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Switch Colab runtime to T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

coverage_path = DATASET_TARGET / 'coverage_overall.json'
if coverage_path.exists():
    coverage = json.loads(coverage_path.read_text())
    print('\nCoverage summary:')
    print(json.dumps(coverage, indent=2))
else:
    print('coverage_overall.json not found; continuing.')

for stem in ['weekly_event_dataset', 'price_features', 'sec_filings_text', 'finnhub_news', 'yahoo_news']:
    exists = (DATASET_TARGET / f'{stem}.csv').exists() or (DATASET_TARGET / f'{stem}.parquet').exists()
    print(f'{stem}:', 'ok' if exists else 'missing')

## 5. Build the chronological split

This writes `artifacts/splits/sp500_500_2yr` and records full/train/test class balance.

In [ ]:
import json
import subprocess
from pathlib import Path

split_dir = Path('artifacts') / 'splits' / DATASET_NAME
input_dir = Path('data') / DATASET_NAME

prepare_cmd = [
    'python3',
    'scripts/prepare_train_test_split.py',
    '--input-dir', str(input_dir),
    '--dataset-name', DATASET_NAME,
]
print('Running:', ' '.join(prepare_cmd))
subprocess.run(prepare_cmd, check=True)

split_metadata = json.loads((split_dir / 'split_metadata.json').read_text())
for key in [
    'total_rows', 'train_rows', 'test_rows',
    'train_start', 'train_end', 'test_start', 'test_end',
    'label_column', 'full_positive_rate', 'train_positive_rate', 'test_positive_rate',
    'full_class_counts', 'train_class_counts', 'test_class_counts',
]:
    print(f'{key}: {split_metadata.get(key)}')

## 6. Train FinBERT multimodal attention

Safe T4 defaults:

- `MAX_LENGTH = 512`
- `BATCH_SIZE = 8`
- `UNFREEZE_FINBERT = False`

If you get CUDA out-of-memory, set `BATCH_SIZE = 4` or `2` in the config cell and rerun from this cell.

In [ ]:
import subprocess
from pathlib import Path

output_dir = Path('artifacts') / 'experiments' / 'finbert_multimodal_attention' / f'{DATASET_NAME}_colab'
split_dir = Path('artifacts') / 'splits' / DATASET_NAME
input_dir = Path('data') / DATASET_NAME

train_cmd = [
    'python3',
    'scripts/train_multimodal_attention.py',
    '--split-dir', str(split_dir),
    '--dataset-name', DATASET_NAME,
    '--input-dir', str(input_dir),
    '--output-dir', str(output_dir),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--max-length', str(MAX_LENGTH),
    '--learning-rate', str(LEARNING_RATE),
    '--checkpoint-metric', CHECKPOINT_METRIC,
    '--threshold-metric', THRESHOLD_METRIC,
    '--alpha-trade-objective', ALPHA_TRADE_OBJECTIVE,
]

if INCLUDE_TICKER:
    train_cmd.append('--include-ticker')
if TRAIN_ON_TEXT_ONLY:
    train_cmd.append('--train-on-text-only')
if LOCAL_FILES_ONLY:
    train_cmd.append('--local-files-only')
if UNFREEZE_FINBERT:
    train_cmd.append('--unfreeze-finbert')

print('Running:')
print(' '.join(train_cmd))
subprocess.run(train_cmd, check=True)

## 7. Inspect results

In [ ]:
import json
from pathlib import Path

run_dir = Path('artifacts') / 'experiments' / 'finbert_multimodal_attention' / f'{DATASET_NAME}_colab'
metrics = json.loads((run_dir / 'metrics.json').read_text())
metadata = json.loads((run_dir / 'metadata.json').read_text())
trading_summary = json.loads((run_dir / 'trading_summary.json').read_text())

print('Metrics:')
print(json.dumps(metrics, indent=2))

print('\nKey metadata:')
for key in [
    'model_name', 'dataset_name', 'label_column',
    'trading_mode', 'signal_regime', 'portfolio_type',
    'train_rows', 'val_rows', 'test_rows',
    'full_positive_rate', 'train_positive_rate', 'test_positive_rate',
    'text_coverage_train', 'text_coverage_test',
    'threshold', 'alpha_trade_threshold', 'best_checkpoint_epoch',
]:
    print(f'{key}: {metadata.get(key)}')

print('\nTrading summary:')
print(json.dumps(trading_summary, indent=2))

## 8. Back up artifacts to Drive

This copies the whole `artifacts/` folder to `MyDrive/CIS2450/finbert_colab_artifacts` and also creates a zip.

In [ ]:
import shutil
from pathlib import Path

src_artifacts = Path('artifacts')
dst_artifacts = ARTIFACT_BACKUP_DIR / f'{DATASET_NAME}_finbert_colab_artifacts'

if dst_artifacts.exists():
    shutil.rmtree(dst_artifacts)
shutil.copytree(src_artifacts, dst_artifacts)

zip_base = ARTIFACT_BACKUP_DIR / f'{DATASET_NAME}_finbert_colab_artifacts'
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=dst_artifacts)

print('Artifacts folder copied to:', dst_artifacts)
print('Artifacts zip written to:', zip_path)

## Troubleshooting

### `max_length=1024` fails

Expected. `ProsusAI/finbert` is BERT-based and only supports 512 tokens. The tokenizer truncates text; it does not compress it.

### CUDA out of memory

Use `BATCH_SIZE = 4` or `BATCH_SIZE = 2`. Keep `MAX_LENGTH = 512`.

### First run cannot find FinBERT weights

Set `LOCAL_FILES_ONLY = False`. After the model downloads into `HF_CACHE_DIR`, repeat runs can use `LOCAL_FILES_ONLY = True`.

### Yahoo rows are zero

Yahoo RSS likely rate-limited the downloader. The model still trains on SEC + Finnhub text. For Yahoo, regenerate later with fewer news workers.